# Assignment 8: Hybrid Agentic Workflows using LangGraph

In [1]:
pip install langgraph

Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import TypedDict, Dict, Any, List
from langgraph.graph import StateGraph, END


# COMMON STATE


class AgentState(TypedDict):
    domain: str
    input_text: str
    extracted_data: Dict[str, Any]
    severity: str
    backend_results: Dict[str, Any]
    route: str
    final_output: str


# TELECOM WORKFLOW NODES

def telecom_understanding(state: AgentState) -> AgentState:
    complaint = state["input_text"].lower()

    issue_type = "general network issue"
    severity = "low"
    affected_scope = "individual customer"

    if "no signal" in complaint or "outage" in complaint or "entire area" in complaint:
        issue_type = "tower outage"
        severity = "high"
        affected_scope = "entire region"
    elif "slow" in complaint or "internet" in complaint:
        issue_type = "slow internet"
        severity = "medium"
    elif "sim" in complaint:
        issue_type = "SIM/device issue"
        severity = "medium"
    elif "charged" in complaint or "billing" in complaint:
        issue_type = "billing network dispute"
        severity = "medium"

    state["extracted_data"] = {
        "issue_type": issue_type,
        "affected_scope": affected_scope
    }
    state["severity"] = severity
    return state


def telecom_backend_check(state: AgentState) -> AgentState:
    issue = state["extracted_data"]["issue_type"]

    results = {
        "tower_health": "normal",
        "crm_ticket": "created",
        "billing_status": "no billing issue",
        "device_status": "compatible",
        "outage_api": "no outage"
    }

    if issue == "tower outage":
        results["tower_health"] = "tower down"
        results["outage_api"] = "active outage detected"
    elif issue == "billing network dispute":
        results["billing_status"] = "billing dispute found"
    elif issue == "SIM/device issue":
        results["device_status"] = "SIM/device issue detected"

    state["backend_results"] = results
    return state


def telecom_router(state: AgentState) -> str:
    issue = state["extracted_data"]["issue_type"]
    severity = state["severity"]

    if severity == "high":
        return "telecom_escalation_agent"
    elif issue == "tower outage":
        return "network_ops_agent"
    elif issue == "billing network dispute":
        return "billing_agent"
    elif issue == "SIM/device issue":
        return "device_support_agent"
    else:
        return "general_network_agent"


def telecom_escalation_agent(state: AgentState) -> AgentState:
    state["route"] = "Escalation Agent"
    return state


def network_ops_agent(state: AgentState) -> AgentState:
    state["route"] = "Network Ops Agent"
    return state


def billing_agent(state: AgentState) -> AgentState:
    state["route"] = "Billing Agent"
    return state


def device_support_agent(state: AgentState) -> AgentState:
    state["route"] = "Device Support Agent"
    return state


def general_network_agent(state: AgentState) -> AgentState:
    state["route"] = "General Network Agent"
    return state


def telecom_resolution(state: AgentState) -> AgentState:
    state["final_output"] = f"""
TELECOM INCIDENT SUMMARY

Issue Type: {state['extracted_data']['issue_type']}
Severity: {state['severity']}
Affected Scope: {state['extracted_data']['affected_scope']}
Assigned To: {state['route']}

Backend Findings:
{state['backend_results']}

Customer Response:
We have analyzed your complaint and routed it to the {state['route']}.
Our team will investigate and update you soon.

Internal Technician Notes:
Check backend results and prioritize based on severity.

Estimated Resolution Time:
{'2-4 hours' if state['severity'] == 'high' else '24 hours'}
"""
    return state


# HEALTHCARE WORKFLOW NODES

def healthcare_understanding(state: AgentState) -> AgentState:
    symptoms = state["input_text"].lower()

    severity = "low"
    condition_type = "general condition"

    if "chest pain" in symptoms or "shortness of breath" in symptoms:
        severity = "high"
        condition_type = "cardiac emergency"
    elif "dizziness" in symptoms or "blurred vision" in symptoms:
        severity = "high"
        condition_type = "neurological symptoms"
    elif "rash" in symptoms or "medication" in symptoms:
        severity = "medium"
        condition_type = "medication reaction"
    elif "fever" in symptoms or "headache" in symptoms:
        severity = "medium"
        condition_type = "common illness"

    state["extracted_data"] = {
        "condition_type": condition_type,
        "symptoms": symptoms
    }
    state["severity"] = severity
    return state


def healthcare_retrieval(state: AgentState) -> AgentState:
    condition = state["extracted_data"]["condition_type"]

    results = {
        "medical_guidelines": "basic triage guidance retrieved",
        "drug_interaction": "no major interaction found",
        "specialist_recommendation": "general physician",
        "historical_cases": "similar cases found"
    }

    if condition == "cardiac emergency":
        results["specialist_recommendation"] = "cardiology"
    elif condition == "neurological symptoms":
        results["specialist_recommendation"] = "neurology"
    elif condition == "medication reaction":
        results["specialist_recommendation"] = "pharmacy review"

    state["backend_results"] = results
    return state


def healthcare_router(state: AgentState) -> str:
    condition = state["extracted_data"]["condition_type"]
    severity = state["severity"]

    if severity == "high":
        return "emergency_agent"
    elif condition == "cardiac emergency":
        return "cardiology_agent"
    elif condition == "neurological symptoms":
        return "neuro_agent"
    elif condition == "medication reaction":
        return "pharmacy_agent"
    else:
        return "general_physician_agent"


def emergency_agent(state: AgentState) -> AgentState:
    state["route"] = "Emergency Escalation Agent"
    return state


def cardiology_agent(state: AgentState) -> AgentState:
    state["route"] = "Cardiology Agent"
    return state


def neuro_agent(state: AgentState) -> AgentState:
    state["route"] = "Neuro Specialist Agent"
    return state


def pharmacy_agent(state: AgentState) -> AgentState:
    state["route"] = "Pharmacy Review Agent"
    return state


def general_physician_agent(state: AgentState) -> AgentState:
    state["route"] = "General Physician Agent"
    return state


def healthcare_resolution(state: AgentState) -> AgentState:
    state["final_output"] = f"""
HEALTHCARE TRIAGE SUMMARY

Condition Type: {state['extracted_data']['condition_type']}
Severity: {state['severity']}
Assigned To: {state['route']}

Retrieved Knowledge:
{state['backend_results']}

Patient Instructions:
Please consult the recommended department. 
If symptoms worsen, visit emergency care immediately.

Recommended Tests:
Basic vitals check, blood test, and specialist-specific diagnosis.

Emergency Notification:
{'Emergency desk notified' if state['severity'] == 'high' else 'Not required'}
"""
    return state


# FINANCE WORKFLOW NODES

def finance_analysis(state: AgentState) -> AgentState:
    text = state["input_text"].lower()

    risk = "medium"
    profile = "average applicant"

    if "credit score 800" in text or "high salary" in text:
        risk = "low"
        profile = "excellent profile"
    elif "missing document" in text:
        risk = "medium"
        profile = "missing documents"
    elif "fraud" in text:
        risk = "high"
        profile = "fraud risk"
    elif "high debt" in text:
        risk = "high"
        profile = "high debt ratio"

    state["extracted_data"] = {
        "applicant_profile": profile,
        "risk_level": risk
    }
    state["severity"] = risk
    return state


def finance_verification(state: AgentState) -> AgentState:
    profile = state["extracted_data"]["applicant_profile"]

    results = {
        "credit_bureau": "verified",
        "fraud_check": "clear",
        "kyc": "verified",
        "employment": "verified",
        "debt_to_income": "acceptable"
    }

    if profile == "fraud risk":
        results["fraud_check"] = "high fraud probability"
    elif profile == "missing documents":
        results["kyc"] = "documents missing"
    elif profile == "high debt ratio":
        results["debt_to_income"] = "too high"

    state["backend_results"] = results
    return state


def finance_router(state: AgentState) -> str:
    profile = state["extracted_data"]["applicant_profile"]
    risk = state["severity"]

    if profile == "fraud risk":
        return "fraud_agent"
    elif profile == "excellent profile":
        return "fast_track_agent"
    elif profile == "missing documents":
        return "clarification_agent"
    elif profile == "high debt ratio":
        return "rejection_agent"
    elif risk == "medium":
        return "manual_review_agent"
    else:
        return "fast_track_agent"


def fraud_agent(state: AgentState) -> AgentState:
    state["route"] = "Fraud Investigation Agent"
    return state


def fast_track_agent(state: AgentState) -> AgentState:
    state["route"] = "Fast-track Approval Agent"
    return state


def clarification_agent(state: AgentState) -> AgentState:
    state["route"] = "Clarification Agent"
    return state


def rejection_agent(state: AgentState) -> AgentState:
    state["route"] = "Rejection Recommendation Agent"
    return state


def manual_review_agent(state: AgentState) -> AgentState:
    state["route"] = "Manual Review Agent"
    return state


def finance_resolution(state: AgentState) -> AgentState:
    state["final_output"] = f"""
FINANCE LOAN DECISION SUMMARY

Applicant Profile: {state['extracted_data']['applicant_profile']}
Risk Level: {state['severity']}
Assigned To: {state['route']}

Verification Results:
{state['backend_results']}

Decision Rationale:
The application has been routed based on fraud risk, creditworthiness, KYC, and debt ratio.

Recommended Loan Amount:
{'Approved as requested' if state['route'] == 'Fast-track Approval Agent' else 'Subject to review'}

Interest Slab:
{'Low interest slab' if state['severity'] == 'low' else 'Standard/high-risk slab'}

Applicant Notification:
Your loan application has been processed and routed to the appropriate team.
"""
    return state


# BUILD TELECOM GRAPH

def build_telecom_graph():
    graph = StateGraph(AgentState)

    graph.add_node("understand", telecom_understanding)
    graph.add_node("backend_check", telecom_backend_check)
    graph.add_node("telecom_escalation_agent", telecom_escalation_agent)
    graph.add_node("network_ops_agent", network_ops_agent)
    graph.add_node("billing_agent", billing_agent)
    graph.add_node("device_support_agent", device_support_agent)
    graph.add_node("general_network_agent", general_network_agent)
    graph.add_node("resolution", telecom_resolution)

    graph.set_entry_point("understand")
    graph.add_edge("understand", "backend_check")

    graph.add_conditional_edges(
        "backend_check",
        telecom_router,
        {
            "telecom_escalation_agent": "telecom_escalation_agent",
            "network_ops_agent": "network_ops_agent",
            "billing_agent": "billing_agent",
            "device_support_agent": "device_support_agent",
            "general_network_agent": "general_network_agent"
        }
    )

    graph.add_edge("telecom_escalation_agent", "resolution")
    graph.add_edge("network_ops_agent", "resolution")
    graph.add_edge("billing_agent", "resolution")
    graph.add_edge("device_support_agent", "resolution")
    graph.add_edge("general_network_agent", "resolution")
    graph.add_edge("resolution", END)

    return graph.compile()


# BUILD HEALTHCARE GRAPH

def build_healthcare_graph():
    graph = StateGraph(AgentState)

    graph.add_node("understand", healthcare_understanding)
    graph.add_node("retrieval", healthcare_retrieval)
    graph.add_node("emergency_agent", emergency_agent)
    graph.add_node("cardiology_agent", cardiology_agent)
    graph.add_node("neuro_agent", neuro_agent)
    graph.add_node("pharmacy_agent", pharmacy_agent)
    graph.add_node("general_physician_agent", general_physician_agent)
    graph.add_node("resolution", healthcare_resolution)

    graph.set_entry_point("understand")
    graph.add_edge("understand", "retrieval")

    graph.add_conditional_edges(
        "retrieval",
        healthcare_router,
        {
            "emergency_agent": "emergency_agent",
            "cardiology_agent": "cardiology_agent",
            "neuro_agent": "neuro_agent",
            "pharmacy_agent": "pharmacy_agent",
            "general_physician_agent": "general_physician_agent"
        }
    )

    graph.add_edge("emergency_agent", "resolution")
    graph.add_edge("cardiology_agent", "resolution")
    graph.add_edge("neuro_agent", "resolution")
    graph.add_edge("pharmacy_agent", "resolution")
    graph.add_edge("general_physician_agent", "resolution")
    graph.add_edge("resolution", END)

    return graph.compile()


# BUILD FINANCE GRAPH

def build_finance_graph():
    graph = StateGraph(AgentState)

    graph.add_node("analysis", finance_analysis)
    graph.add_node("verification", finance_verification)
    graph.add_node("fraud_agent", fraud_agent)
    graph.add_node("fast_track_agent", fast_track_agent)
    graph.add_node("clarification_agent", clarification_agent)
    graph.add_node("rejection_agent", rejection_agent)
    graph.add_node("manual_review_agent", manual_review_agent)
    graph.add_node("resolution", finance_resolution)

    graph.set_entry_point("analysis")
    graph.add_edge("analysis", "verification")

    graph.add_conditional_edges(
        "verification",
        finance_router,
        {
            "fraud_agent": "fraud_agent",
            "fast_track_agent": "fast_track_agent",
            "clarification_agent": "clarification_agent",
            "rejection_agent": "rejection_agent",
            "manual_review_agent": "manual_review_agent"
        }
    )

    graph.add_edge("fraud_agent", "resolution")
    graph.add_edge("fast_track_agent", "resolution")
    graph.add_edge("clarification_agent", "resolution")
    graph.add_edge("rejection_agent", "resolution")
    graph.add_edge("manual_review_agent", "resolution")
    graph.add_edge("resolution", END)

    return graph.compile()


# RUN WORKFLOW

def run_workflow(domain: str, user_input: str):
    initial_state: AgentState = {
        "domain": domain,
        "input_text": user_input,
        "extracted_data": {},
        "severity": "",
        "backend_results": {},
        "route": "",
        "final_output": ""
    }

    if domain.lower() == "telecom":
        app = build_telecom_graph()
    elif domain.lower() == "healthcare":
        app = build_healthcare_graph()
    elif domain.lower() == "finance":
        app = build_finance_graph()
    else:
        raise ValueError("Invalid domain. Choose telecom, healthcare, or finance.")

    result = app.invoke(initial_state)
    return result


# OPTIONAL: VISUALIZE GRAPH

def save_graph_image(app, filename="workflow.png"):
    try:
        png_data = app.get_graph().draw_mermaid_png()
        with open(filename, "wb") as f:
            f.write(png_data)
        print(f"Graph saved as {filename}")
    except Exception as e:
        print("Graph visualization failed.")
        print("Install dependencies if needed:")
        print("pip install pygraphviz")
        print(e)


# ============================================================
# MAIN PROGRAM
# ============================================================

if __name__ == "__main__":

    print("Choose Domain:")
    print("1. Telecom")
    print("2. Healthcare")
    print("3. Finance")

    choice = input("Enter choice 1/2/3: ")

    if choice == "1":
        domain = "telecom"
        user_input = input("Enter telecom complaint: ")

    elif choice == "2":
        domain = "healthcare"
        user_input = input("Enter patient symptoms: ")

    elif choice == "3":
        domain = "finance"
        user_input = input("Enter loan applicant details: ")

    else:
        print("Invalid choice")
        exit()

    output = run_workflow(domain, user_input)

    print("\n================ FINAL OUTPUT ================")
    print(output["final_output"])

C:\ProgramData\anaconda3\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Choose Domain:
1. Telecom
2. Healthcare
3. Finance


Enter choice 1/2/3:  1
Enter telecom complaint:  high



================ FINAL OUTPUT ================

TELECOM INCIDENT SUMMARY

Issue Type: general network issue
Severity: low
Affected Scope: individual customer
Assigned To: General Network Agent

Backend Findings:
{'tower_health': 'normal', 'crm_ticket': 'created', 'billing_status': 'no billing issue', 'device_status': 'compatible', 'outage_api': 'no outage'}

Customer Response:
We have analyzed your complaint and routed it to the General Network Agent.
Our team will investigate and update you soon.

Internal Technician Notes:
Check backend results and prioritize based on severity.

Estimated Resolution Time:
24 hours

